# 1. Mount gdrive and import libs

In [1]:
from google.colab import drive
drive.mount('/content/gdrive')
import pandas as pd
import numpy as np
import re
from pathlib import Path
from typing import Literal
import matplotlib.pyplot as plt

from skimage.filters import gaussian, threshold_otsu
from skimage.segmentation import *
import skimage.io as skio
from skimage.feature import graycomatrix, graycoprops
import skimage.morphology as skmor
import skimage.measure as skmea
from skimage.morphology import (
	disk,
	binary_opening,
	binary_closing,
	remove_small_objects,
	remove_small_holes
)

from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, Subset

!pip install xgboost
from sklearn.metrics import accuracy_score,classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


Mounted at /content/gdrive


# 2.Unzip dataset zip file

In [2]:
#!unzip "/content/gdrive/MyDrive/BMET5933/WEEK_8/hep2img.zip" -d "/content/gdrive/MyDrive/BMET5933/WEEK_8"
#!unzip "/content/gdrive/MyDrive/BMET5933/WEEK_8/hep2img_large.zip" -d "/content/gdrive/MyDrive/BMET5933/WEEK_8"

# 3.Dataset class definition:
Chatgpt5.4 is used for designing, debugging and adding boundary condition structure of this dataset class.

In [3]:
class Hep2Dataset(Dataset):
	def __init__(self, dataset_root, csv_path, transform=None):
		"""
		Initialize the dataset.

		Args:
			dataset_root (str or Path): Root folder of the image dataset.
			csv_path (str or Path): CSV file containing image file names and labels.
			transform (callable, optional): Transform to apply to each image.
		"""
		self.dataset_root = Path(dataset_root)
		self.csv_path = Path(csv_path)
		self.transform = transform

		# Read all samples once and store them as a unified list
		self.samples = self.read_label(self.csv_path)

	def read_label(self, csv_path):
		"""
		Read the CSV file and build a list of (image_path, label).

		Args:
			csv_path (Path): Path to the CSV label file.

		Returns:
			list: A list of tuples, where each tuple is (image_path, label).
		"""
		df = pd.read_csv(csv_path)

		samples = []
		for _, row in df.iterrows():
			img_name = row["file"]
			label = int(row["class"])  # Convert label to integer
			mask_path = self.dataset_root / f'{str(img_name).zfill(3)}_Mask.png'
			img_path = self.dataset_root / f'{str(img_name).zfill(3)}.png'
			samples.append((img_path, mask_path, label))

		return samples

	def __len__(self):
		"""
		Return the total number of samples in the dataset.
		"""
		return len(self.samples)

	def __getitem__(self, idx):
		"""
		Get one sample by index.

		Args:
			idx (int): Index of the sample.

		Returns:
			tuple: (image, mask, label)
		"""
		img_path, mask_path, label = self.samples[idx]
		img = skio.imread(img_path)
		mask = skio.imread(mask_path)

		if self.transform is not None:
			img = self.transform(img)

		return img, mask, label

# 4.Shape feature extractor:
1.Chatgpt5.4 used for designing shape feature extraction class.

2.Here I used circularity, solidity and eccentricity beacuse we don't have metadata like what we have in DICOM format. So some features like perimeter and area, we cannnot get.

In [4]:
class ShapeFeatureExtractor:
	def __init__(self):
		return

	def _get_largest_region(self, mask):
		"""
		Get the largest connected component from a mask.

		Args:
			mask (ndarray): Input mask.

		Returns:
			regionprops object or None: Largest region if exists, else None.
		"""
		mask = mask > 0
		labeled_mask = skmea.label(mask)

		regions = skmea.regionprops(labeled_mask)
		if len(regions) == 0:
			return None

		largest_region = max(regions, key=lambda x: x.area)
		return largest_region

	def _get_solidity_from_region(self, region):
		"""
		Calculate solidity from a regionprops object.

		Args:
			region: regionprops object or None.
		Returns:
			float: Solidity value. Returns 0.0 if no valid region exists.
		"""
		if region is None:
			return 0.0

		return float(region.solidity)

	def _get_eccentricity_from_region(self, region):
		"""
		Calculate eccentricity from a regionprops object.

		Args:
			region: regionprops object or None.
		Returns:
			float: Eccentricity value. Returns 0.0 if no valid region exists.
		"""
		if region is None:
			return 0.0

		return float(region.eccentricity)

	def _get_circularity_from_region(self, region):
		"""
		Calculate circularity from a regionprops object.

		Formula:
			circularity = 4 * pi * area / perimeter^2

		Args:
			region: regionprops object or None.
		Returns:
			float: Circularity value. Returns 0.0 if no valid region exists
				   or perimeter is zero.
		"""
		if region is None:
			return 0.0

		area = region.area
		perimeter = region.perimeter

		if perimeter == 0:
			return 0.0

		circularity = 4.0 * np.pi * area / (perimeter ** 2)
		return float(circularity)


	def get_shape_features(self, mask):
		"""
		Extract all shape features from a single mask.

		Args:
			mask (ndarray): Input mask.

		Returns:
			dict: Dictionary containing shape features.
		"""
		largest_region = self._get_largest_region(mask)
		single_feature = {
			"circularity": self._get_circularity_from_region(largest_region),
			"solidity": self._get_solidity_from_region(largest_region),
			"eccentricity": self._get_eccentricity_from_region(largest_region),
		}
		return single_feature


# 5.GLCM feature exatractor:

In [5]:
class GlcmFeatureExtractor:
	def __init__(self):
		"""
		Initialize the GLCM feature extractor.

		Args:
			image (ndarray): Input image.
			label (int): Corresponding label.
			mask (ndarray): Corresponding mask.
		"""

		# glcm configuration
		self.distance = 1
		self.angle = 0
		self.levels = 32
		self.symmetric = False
		self.normed = False

	def get_quantized_image(self, image):
		"""
		Get the gray-level image.

		Args:
			image (ndarray): Input image.
		Returns:
			ndarray: Gray-level image.
		"""
		quantized_gray_image = np.floor(image / (self.levels)).astype(np.uint8)
		return quantized_gray_image

	def get_masked_gray_image(self, image, mask):
		"""
		Get the gray-level image with the background masked out.

		Args:
			image (ndarray): Input image.
			mask (ndarray): Input mask.
		Returns:
			ndarray: Gray-level image with background masked out.
		"""
		background_mask  = ~(mask > 0)
		gray_image = self.get_quantized_image(image)

		# only keep foreground region
		masked_image = gray_image.copy()
		masked_image[background_mask] = 0
		return masked_image

	def get_glcm_features(self, image, mask):
		glcm_features = {}

		# get masked gray image first
		masked_image = self.get_masked_gray_image(image, mask)

		# get GLCM matrix
		glcm = graycomatrix(
					masked_image.astype(np.uint8),
					distances=[self.distance],
					angles=[self.angle],
					levels=self.levels,
					symmetric=self.symmetric,
					normed=self.normed
					)
		# extract GLCM features
		glcm_contrast = graycoprops(glcm, prop='contrast')[0, 0]
		glcm_dissimilarity = graycoprops(glcm, prop='dissimilarity')[0, 0]
		glcm_homogeneity = graycoprops(glcm, prop='homogeneity')[0, 0]
		glcm_energy = graycoprops(glcm, prop='energy')[0, 0]
		glcm_correlation = graycoprops(glcm, prop='correlation')[0, 0]

		# store features in a dictionary
		glcm_features = {
			"contrast": glcm_contrast,
			"dissimilarity": glcm_dissimilarity,
			"homogeneity": glcm_homogeneity,
			"energy": glcm_energy,
			"correlation": glcm_correlation
		}

		return glcm_features


# 6.Gray intensity feature extractor:

In [6]:
class ColorFeatureExtractor:
	def __init__(self):
		return

	def get_masked_gray_image(self, image, mask):
		"""
		Get the grayscale image with the background masked out.
		"""
		background_mask = ~(mask > 0)
		masked_image = image.copy()
		masked_image[background_mask] = 0
		return masked_image

	def get_gray_max_min(self, image, mask):
		"""
		Calculate the max and min pixel intensities in the masked image.
		"""
		masked_image = self.get_masked_gray_image(image, mask)
		max_gray = masked_image[mask > 0].max()
		min_gray = masked_image[mask > 0].min()
		return max_gray, min_gray

	def get_gray_avg(self, image, mask):
		"""
		Calculate the average pixel intensity in the masked image.
		"""
		masked_image = self.get_masked_gray_image(image, mask)
		avg_gray = masked_image[mask > 0].mean()
		return avg_gray

	def get_gray_std(self, image, mask):
		"""
		Calculate the standard deviation of pixel intensities in the masked image.
		"""
		masked_image = self.get_masked_gray_image(image, mask)
		std_gray = masked_image[mask > 0].std()
		return std_gray

	def get_gray_histogram(self, image, mask):
		"""
		Calculate the grayscale histogram of pixel intensities in the masked image.
		"""
		masked_image = self.get_masked_gray_image(image, mask)
		histogram, bin_edges = np.histogram(masked_image[mask > 0], bins=256, range=(0, 256))
		return histogram, bin_edges

	def get_gray_intensity_features(self, image, mask):
		"""
		Calculate a set of grayscale intensity features for a masked image.
		"""
		gray_features = {}
		gray_features["avg_intensity"] = self.get_gray_avg(image, mask)
		gray_features["std_intensity"] = self.get_gray_std(image, mask)
		gray_features["max_intensity"], gray_features["min_intensity"] = self.get_gray_max_min(image, mask)
		return gray_features


# 7.Featrue extraction process and Dataset initialization

In [7]:
# Dataset initialization
dataset = Hep2Dataset(
	dataset_root="/content/gdrive/MyDrive/BMET5933/WEEK_8/hep2img_large",
	csv_path="/content/gdrive/MyDrive/BMET5933/WEEK_8/hep_img_large_gt.csv"
)

In [9]:
# instantiate feature/ color extractors
shape_extractor = ShapeFeatureExtractor()
color_extractor = ColorFeatureExtractor()
glcm_extractor = GlcmFeatureExtractor()

feature_list = []
for i in range(len(dataset)):
	image, mask, label = dataset[i] # Unpack the tuple
	feature = {}
	# store features in a dictionary, and update the dictionary with different types of features
	feature.update(shape_extractor.get_shape_features(mask))
	feature.update(color_extractor.get_gray_intensity_features(image, mask))
	feature.update(glcm_extractor.get_glcm_features(image, mask))
	feature['label'] = label # Use the unpacked label
	feature_list.append(feature)

# convert into dataframe and extract x and y for training
feature_df = pd.DataFrame(feature_list)
x = feature_df.drop(columns=['label'])
y = feature_df['label']



## Feature extraction explanation

In this part, I converted each cell image into one row of numerical features. I used the mask to focus on the cell region instead of the background. The shape features describe the cell boundary and region, the gray intensity features describe the brightness distribution inside the cell, and the GLCM features describe texture patterns. After extracting these features, I stored them in `feature_df`, where each row represents one image sample and the `label` column is the class that the model needs to predict.


In [12]:
# label converting
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# split the dataset into training and testing subsets, ratio is a experience-based choice
train_test_split_ratio = 0.2
x_train, x_test, y_train,y_test = train_test_split(
										x, 
										y_encoded, 
										test_size = 
										train_test_split_ratio, 
										random_state=42, 
										stratify=y_encoded
										)

## Train/test split explanation

I used an 80/20 train/test split. This gives most of the data to the model for training, while still keeping a separate test set to evaluate how well the model works on unseen images. I also used `stratify=y_encoded` because this is a multi-class dataset, and I want the training and test sets to keep a similar class distribution. The `random_state=42` makes the split reproducible, so the result can be checked again with the same data split.


# 8.Classifier training

In [13]:
def evaluate_model(model_name, model, x_train, x_test, y_train, y_test):
    model.fit(x_train, y_train)
    y_pred = model.predict(x_test)

    accuracy = accuracy_score(y_test, y_pred)
    report_dict = classification_report(y_test, y_pred, output_dict=True)

    print(f"{model_name} Accuracy: {accuracy:.4f}")
    print("Classification Report:")
    print(classification_report(
        y_test,
        y_pred,
        target_names=[str(c) for c in label_encoder.classes_]
    ))

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    return {
        "model": model_name,
        "accuracy": accuracy,
        "macro_f1": report_dict["macro avg"]["f1-score"],
        "weighted_f1": report_dict["weighted avg"]["f1-score"]
    }


In [14]:
# Model definition
xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.1,
    objective='multi:softmax',
    eval_metric='merror',
    random_state=42
)

xgb_results = evaluate_model("XGBoost", xgb_model, x_train, x_test, y_train, y_test)


XGBoost Accuracy: 0.8462
Classification Report:
              precision    recall  f1-score   support

           1       0.81      0.72      0.76        18
           2       0.80      0.94      0.86        17
           3       0.86      0.90      0.88        20
           4       0.80      0.80      0.80        20
           5       1.00      0.88      0.93        16

    accuracy                           0.85        91
   macro avg       0.85      0.85      0.85        91
weighted avg       0.85      0.85      0.85        91

Confusion Matrix:
[[13  1  2  2  0]
 [ 1 16  0  0  0]
 [ 2  0 18  0  0]
 [ 0  3  1 16  0]
 [ 0  0  0  2 14]]


## Why I chose XGBoost

I chose XGBoost as the basic classifier because my extracted image features are tabular numerical data, which tree-based models usually handle well. XGBoost builds many decision trees step by step. Each new tree tries to correct the mistakes made by the previous trees, so the final model can learn more complex relationships between shape, intensity, texture features and the image classes. It is also suitable here because the relationship between the features and the HEp-2 staining patterns is probably non-linear.


# 9.Challenge part:

## Evaluation interpretation

The XGBoost model achieved a test accuracy of about 0.8462, which means it correctly classified around 84.62% of the test images. I also used the classification report because accuracy alone does not show how each class performs. Precision shows how many predicted samples of a class are actually correct, recall shows how many real samples of a class are found by the model, and F1-score balances precision and recall. The macro average treats every class equally, while the weighted average considers the number of samples in each class. The confusion matrix shows the detailed prediction errors: values on the diagonal are correct predictions, and off-diagonal values are misclassified samples.


## Challenge 1:

In [15]:
# model initialization, parameters are experience-based choices
# random forest
rf_model = RandomForestClassifier(
	n_estimators=500,
	max_depth=5,
	random_state=42,
    class_weight='balanced'  # Add class_weight to handle class imbalance
)

# SVM with RBF kernel, parameters are experience-based choices
svm_model= Pipeline([
	('scaler', StandardScaler()),
	('svc', SVC(kernel='rbf', random_state=42))
])	



In [16]:
# model evaluation and comparison
results = []

results.append(evaluate_model("XGBoost", xgb_model, x_train, x_test, y_train, y_test))
results.append(evaluate_model("Random Forest", rf_model, x_train, x_test, y_train, y_test))
results.append(evaluate_model("SVM RBF", svm_model, x_train, x_test, y_train, y_test))

comparison_df = pd.DataFrame(results)
comparison_df


XGBoost Accuracy: 0.8462
Classification Report:
              precision    recall  f1-score   support

           1       0.81      0.72      0.76        18
           2       0.80      0.94      0.86        17
           3       0.86      0.90      0.88        20
           4       0.80      0.80      0.80        20
           5       1.00      0.88      0.93        16

    accuracy                           0.85        91
   macro avg       0.85      0.85      0.85        91
weighted avg       0.85      0.85      0.85        91

Confusion Matrix:
[[13  1  2  2  0]
 [ 1 16  0  0  0]
 [ 2  0 18  0  0]
 [ 0  3  1 16  0]
 [ 0  0  0  2 14]]
Random Forest Accuracy: 0.7692
Classification Report:
              precision    recall  f1-score   support

           1       0.65      0.83      0.73        18
           2       0.93      0.76      0.84        17
           3       0.78      0.90      0.84        20
           4       0.65      0.65      0.65        20
           5       1.00      

,model,accuracy,macro_f1,weighted_f1
0,XGBoost,0.846154,0.848191,0.845733
1,Random Forest,0.769231,0.774488,0.771539
2,SVM RBF,0.780220,0.784576,0.782218


In [ ]:
# These are for exporting the notebook as a PDF later if needed
!apt-get update
!sudo apt-get install texlive-xetex texlive-fonts-recommended texlive-plain-generic pandoc
!jupyter nbconvert --to pdf /content/gdrive/MyDrive/BMET5933/WEEK_8/Week_8-9_Li_Code_file_ipynb_draft_version.ipynb

Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Hit:3 https://cli.github.com/packages stable InRelease                         
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Hit:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease     
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease               
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease   
Hit:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease    
Hit:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Fetched 128 kB in 1s (151 kB/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Re